# 05 — 驗證 MNIST ONNX（Pipeline 第二步）

本 Notebook 是 **單元 B-Vis Pipeline 的第二個節點**：確認上一步訓練寫出的 `model.onnx` 可載入並做一次本機推論。

| 用途 | 說明 |
|------|------|
| **Pipeline Editor** | 接在 `04-train-pytorch-mnist.ipynb` **之後**（拉線連接） |
| 與 Dashboard Deploy | 本步**不**部署；Deploy 仍在 Pipeline Run 成功後手動做 |

**讀取路徑**（與 `04` 相同）：
```
/opt/app-root/src/model-storage/mnist-onnx/1/model.onnx
```


In [ ]:
%pip install -q onnxruntime onnx


In [ ]:
from pathlib import Path
import numpy as np
import onnxruntime as ort

MOUNT = Path("/opt/app-root/src/model-storage")
MODEL_DIR = MOUNT / "mnist-onnx"
ONNX_PATH = MODEL_DIR / "1" / "model.onnx"

# 相容：舊 Run 把檔寫在掛載根（PVC 根的 1/ 或 model.onnx）
candidates = [
    ONNX_PATH,
    MOUNT / "1" / "model.onnx",
    MOUNT / "model.onnx",
    Path("/opt/app-root/src/mnist-onnx/1/model.onnx"),  # 更舊的 mount path
    Path("/opt/app-root/src/mnist-onnx/model.onnx"),
]
ONNX_PATH = next((p for p in candidates if p.exists()), ONNX_PATH)
if ONNX_PATH != MODEL_DIR / "1" / "model.onnx" and ONNX_PATH.exists():
    print(f"WARN: 使用相容路徑 {ONNX_PATH}；建議重跑 04 寫入 mnist-onnx/1/model.onnx")

assert ONNX_PATH.exists(), (
    f"找不到 model.onnx — 試過: {', '.join(str(p) for p in candidates)}。"
    "確認 04 Succeeded，且 Data Volume Mount path=/opt/app-root/src/model-storage"
)
print(f"OK: {ONNX_PATH} ({ONNX_PATH.stat().st_size} bytes)")

session = ort.InferenceSession(str(ONNX_PATH), providers=["CPUExecutionProvider"])
input_name = session.get_inputs()[0].name
output_name = session.get_outputs()[0].name
print(f"input={input_name}, output={output_name}")

# 全零影像：僅驗證張量形狀與可推論（非真實手寫辨識準確率）
dummy = np.zeros((1, 1, 28, 28), dtype=np.float32)
out = session.run([output_name], {input_name: dummy})[0]
print(f"output shape={out.shape}, sample logits[:5]={out.reshape(-1)[:5]}")
pred = int(out.reshape(-1).argmax())
print(f"Predicted digit (dummy zeros): {pred}")
print("ONNX validation completed successfully.")


## 下一步：Dashboard 部署 + B-7 驗證

兩個 Pipeline 節點都 **Succeeded** 後：

1. Dashboard → **Deploy model**
2. Name：`mnist-classifier-elyra`；Framework：**ONNX**；Model path：**`mnist-onnx/`**（**不要**填 `/`）
3. Ready 後開啟 **`06-test-mnist-inference.ipynb`**，Run 所有 cell（單元 B-7）

見入門教學單元 B。
